In [5]:
import numpy as np
import pandas as pd
import netCDF4 as nc
from pathlib import Path
from pyproj import Transformer

# Set input file and output directory paths
FILE = Path("/Users/marcel/Desktop/iceberg-motion/outputs/2024-06-14/spireberg_mbes_cleaned_2024-06-14.nc")
OUT_DIR = Path("../data/2024-06-14/file_point_clouds")

# Must match the fill values used in 00_compile_NetCDF.ipynb
FILL_F32 = np.float32(9.96921e+36)
FILL_F64 = np.float64(9.969209968386869e+36)
FILL_I8 = np.int8(-127)
FILL_I32 = np.int32(-2147483647)

# Transformer for reprojection into CRS
transformer = Transformer.from_crs("EPSG:6318", "EPSG:6339", always_xy=True)

In [6]:
# Load netCDF file
ds = nc.Dataset(FILE)

# Get unique ping count
ping_seg = ds.variables["ping_segment"][:]   # (4424,) int8 — 0 through 4

# Get number of source files
src_files = ds.variables["source_file"][:]    # (5,)   string array


# Print information on files, pings, and beams,
print(f"{'Seg':>4}  {'Source File':<35}  {'Pings':>6}  {'Beams (potential)':>18}")
print("-" * 70)
for seg_idx in range(len(ds.dimensions["segment"])):
    n_pings = int(np.sum(ping_seg == seg_idx))
    print(f"  {seg_idx}   {src_files[seg_idx]:<35}  {n_pings:>6,}  {n_pings * 1024:>18,}")

print(f"\nTotal pings: {len(ping_seg):,}")
print(f"Total potential soundings: {len(ping_seg) * 1024:,} (many will be fill/invalid)")


 Seg  Source File                           Pings   Beams (potential)
----------------------------------------------------------------------
  0   20240614_194448_2.txt                   620             634,880
  1   20240614_200001_3.txt                   413             422,912
  2   20240614_200947_4.txt                 1,489           1,524,736
  3   20240614_201828_5.txt                 1,564           1,601,536
  4   20240614_202551_6.txt                   338             346,112

Total pings: 4,424
Total potential soundings: 4,530,176 (many will be fill/invalid)


In [7]:
seg_idx = 0        
mask = ping_seg == seg_idx   
n_pings = int(np.sum(mask))
n_beams = len(ds.dimensions["beam"])
N = n_pings * n_beams

print(f"Segment {seg_idx}: {n_pings} pings × {n_beams} beams = {N:,} sounding slots")

# 1D per-ping data
time_1d = ds.variables["time"][mask]
vsl_lon_1d = ds.variables["vessel_lon"][mask]
vsl_lat_1d = ds.variables["vessel_lat"][mask]
vsl_z_1d = ds.variables["vessel_z"][mask]
vsl_hdg_1d = ds.variables["vessel_heading"][mask]
vsl_pit_1d = ds.variables["vessel_pitch"][mask]
vsl_rol_1d = ds.variables["vessel_roll"][mask]

# 2D per-sounding data
fp_lon_2d = ds.variables["footprint_longitude"][mask, :]
fp_lat_2d = ds.variables["footprint_latitude"][mask, :]
depth_2d = ds.variables["depth"][mask, :]
intens_2d = ds.variables["intensity"][mask, :]
unc_h_2d = ds.variables["uncertainty_horizontal"][mask, :]
unc_v_2d = ds.variables["uncertainty_vertical"][mask, :]

# Flatten 2D to 1D 
fp_lon_flat = fp_lon_2d.flatten()
fp_lat_flat = fp_lat_2d.flatten()
depth_flat = depth_2d.flatten()
intens_flat = intens_2d.flatten()
unc_h_flat = unc_h_2d.flatten()
unc_v_flat = unc_v_2d.flatten()

# Broadcast per-ping arrays to sounding arrays dimensionality
beam_flat = np.tile(np.arange(n_beams, dtype=np.int16), n_pings)
time_flat = np.repeat(time_1d, n_beams)
vlon_flat = np.repeat(vsl_lon_1d, n_beams)
vlat_flat = np.repeat(vsl_lat_1d, n_beams)
vz_flat = np.repeat(vsl_z_1d, n_beams)
vhdg_flat = np.repeat(vsl_hdg_1d, n_beams)
vpit_flat = np.repeat(vsl_pit_1d, n_beams)
vrol_flat = np.repeat(vsl_rol_1d, n_beams)

# Filter out null soundings
valid = depth_flat != FILL_F32

print(f"Valid soundings: {valid.sum():,} / {N:,} ({100*valid.sum()/N:.1f}%)")

# Verify the broadcast is correct
assert beam_flat[0] == 0 and beam_flat[1023] == 1023 and beam_flat[1024] == 0


Segment 0: 620 pings × 1024 beams = 634,880 sounding slots
Valid soundings: 218,451 / 634,880 (34.4%)


In [8]:
# Create output directory if it doesn't exist
OUT_DIR.mkdir(parents=True, exist_ok=True)

# For each original swath file...
for seg_idx in range(len(ds.dimensions["segment"])):

    mask = ping_seg == seg_idx
    n_pings = int(np.sum(mask))
    n_beams = len(ds.dimensions["beam"])

    # Load and flatten
    def rep(arr): return np.repeat(arr.filled(np.nan), n_beams)
    beams = np.tile(np.arange(n_beams, dtype=np.int16), n_pings)

    # Create a mask for all NaN soundings using depth field as proxy for other variables.
    fp_lon = ds.variables["footprint_longitude"][mask, :].flatten().filled(np.nan)
    fp_lat = ds.variables["footprint_latitude"][mask, :].flatten().filled(np.nan)
    depth = ds.variables["depth"][mask, :].flatten().filled(np.nan)
    intens = ds.variables["intensity"][mask, :].flatten().filled(np.nan)
    unc_h = ds.variables["uncertainty_horizontal"][mask, :].flatten().filled(np.nan)
    unc_v = ds.variables["uncertainty_vertical"][mask, :].flatten().filled(np.nan)
    valid = np.isfinite(depth)

    # 1D per-ping arrays broadcast to sounding shape
    vlon_flat = rep(ds.variables["vessel_lon"][mask])
    vlat_flat = rep(ds.variables["vessel_lat"][mask])
    vz_flat = rep(ds.variables["vessel_z"][mask])
    vhdg_flat = rep(ds.variables["vessel_heading"][mask])
    vpit_flat = rep(ds.variables["vessel_pitch"][mask])
    vrol_flat = rep(ds.variables["vessel_roll"][mask])
    time_flat = rep(ds.variables["time"][mask])
    beams = np.tile(np.arange(n_beams, dtype=np.int16), n_pings)

    # Transform into CRS
    fp_east, fp_north  = transformer.transform(fp_lon[valid], fp_lat[valid])
    vsl_east, vsl_north = transformer.transform(vlon_flat[valid], vlat_flat[valid])

    # Mask for valid times
    unix_time = rep(ds.variables["time"][mask])[valid]

    # Fill dataframe
    df = pd.DataFrame({

        # Time
        "UnixTime": unix_time,
        
        # Sounding position
        "FootprintX": fp_east,
        "FootprintY": fp_north,
        "FootprintZ": depth[valid],

        # Acoustic measurements
        "Intensity": intens[valid],
        "UncertaintyVertical": unc_v[valid],
        "UncertaintyHorizontal": unc_h[valid],
        
        # Vessel state
        "VesselX": vsl_east,
        "VesselY": vsl_north,
        "VesselZ": vz_flat[valid],
        "VesselHeading": rep(ds.variables["vessel_heading"][mask])[valid],
        "VesselPitch": rep(ds.variables["vessel_pitch"][mask])[valid],
        "VesselRoll": rep(ds.variables["vessel_roll"][mask])[valid],
    })

    src_name = src_files[seg_idx].replace(".txt", "")
    out_path = OUT_DIR / f"file_{seg_idx:02d}.csv"
    df.to_csv(out_path, index=False)

    print(f"File {seg_idx} to {out_path.name} ({len(df):,} soundings)")

ds.close()
print("\nDone.")


File 0 to file_00.csv (218,451 soundings)
File 1 to file_01.csv (218,879 soundings)
File 2 to file_02.csv (347,490 soundings)
File 3 to file_03.csv (528,458 soundings)
File 4 to file_04.csv (89,712 soundings)

Done.
